# Fundamentals 01 - Tool API

**Historia:** antes de hablar de agentes, entendemos la unidad mínima de Agentic Systems: una `Tool`.

En `2.4.3` además materializamos el contrato alrededor de esa tool: input/output Pydantic, expectativa de ejecución y policy reutilizable para quien la convierta en agente.

In [ ]:
import agentic_systems as lab
from pydantic import BaseModel

PRETTY = False  # Cambia a True para usar Rich; False imprime texto plano estable y reproducible.

## Escenario did?ctico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: sólo representa la sección `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario did?ctico · visible")


## 1) Crear una tool rápida con `@lab.tool`

Aquí no hay agente todavía. Sólo hay una función Python convertida en una `Tool` ejecutable.

In [ ]:
@lab.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos números y regresa output estructurado."""
    return {
        "operation": "sumar",
        "result": a + b,
        "explanation": f"{a} + {b} = {a + b}",
    }

quick_result = sumar.run({"a": 17, "b": 25})
lab.human_result(
    quick_result,
    title="Human result · tool rápida",
    expected_tools=lab.expect.exactly("sumar"),
    pretty=PRETTY,
)

## 2) Crear una tool explícita con schema

Usa `lab.Tool(...)` cuando quieras dejar el contrato de entrada/salida visible desde el objeto.

In [ ]:
class TwoNumbers(BaseModel):
    a: int
    b: int


class OperationOutput(BaseModel):
    operation: str
    result: int
    explanation: str


def restar_fn(payload: TwoNumbers) -> OperationOutput:
    value = payload.a - payload.b
    return OperationOutput(operation="restar", result=value, explanation=f"{payload.a} - {payload.b} = {value}")


restar = lab.Tool(
    restar_fn,
    name="restar",
    description="Resta dos números y devuelve salida estructurada.",
    input=TwoNumbers,
    output=OperationOutput,
)

contract_result = restar.run({"a": 50, "b": 8})
lab.human_result(contract_result, title="Human result · tool con Pydantic", expected_tools=lab.expect.exactly("restar"), pretty=PRETTY)

## 3) Materializar contrato + policy

El contrato responde **qué debe pasar**. La policy responde **con qué límites debe ejecutarse**.

Este spec no ejecuta nada por sí solo; es metadata validable y reusable para agentes, sistemas, skills o evals.

In [ ]:
sumar_spec = lab.ContractPolicySpec(
    name="fundamentals.sumar_once",
    description="Debe llamar sumar exactamente una vez y conservar salida exitosa.",
    contract=lab.AgentContract(
        must_call=["sumar"],
        tool_expectation=lab.expect.exactly("sumar"),
        completion="when_required_tools_satisfied",
        failure_policy="no_unresolved",
        expected_tool_outputs={"sumar": {"operation": "sumar", "result": 42}},
    ),
    policy=lab.RunPolicy(
        max_turns=4,
        max_tool_calls=1,
        temperature=0.0,
        tool_choice="sumar",
        finalize="after_required_tools",
    ),
    tags=["fundamentals", "tool", "contract"],
)

available_tools = [sumar.name, restar.name]
lab.show({
    "spec": sumar_spec.describe(),
    "static_check": sumar_spec.check(available_tools=available_tools).to_dict(),
})

## 4) Validación declarativa antes de ejecutar

Aquí provocamos un contrato imposible para ver el error sin llamar ningún modelo ni tool.

In [ ]:
impossible = lab.ContractPolicySpec(
    name="fundamentals.invalid_budget",
    contract=lab.AgentContract(must_call=["sumar", "restar"]),
    policy=lab.RunPolicy(max_tool_calls=1),
)

lab.show(impossible.check(available_tools=available_tools).to_dict())

## 5) Resolver el escenario did?ctico sólo con tools

Aquí no hay parser ni agente. Leemos el prompt y ejecutamos las operaciones con tools, una por una.

La parte didáctica es ver el patrón:

```text
Tool.run(input) → salida estructurada → siguiente Tool.run(...)
```


In [ ]:
@lab.tool
def multiplicar(a: int, b: int) -> dict:
    """Multiplica dos números."""
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a} × {b} = {a * b}"}


@lab.tool
def dividir(a: int, b: int) -> dict:
    """Divide dos números."""
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    result = int(value) if value.is_integer() else value
    return {"operation": "dividir", "result": result, "explanation": f"{a} ÷ {b} = {result}"}


# Directo del prompt del usuario:
# "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
step_1 = sumar.run({"a": 10, "b": 20})
step_2 = restar.run({"a": step_1.data["result"], "b": 9})
step_3 = multiplicar.run({"a": step_2.data["result"], "b": 4})
step_4 = dividir.run({"a": step_3.data["result"], "b": 2})

procedure = [step.data["explanation"] for step in [step_1, step_2, step_3, step_4]]
answer = {
    "procedimiento": procedure,
    "resultado_final": step_4.data["result"],
}

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
    "respuesta_materializada": answer,
})


## Lo importante

- `Tool` valida input/output.
- `AgentContract` declara expectativas de comportamiento.
- `RunPolicy` declara límites de ejecución.
- `ContractPolicySpec` empaqueta ambas cosas para reuso.

## Coverage API de este notebook

Esta tabla deja explícito qué parte de Agentic Systems queda materializada aquí.

In [ ]:
api_coverage = [
    {
        "api": "@lab.tool",
        "description": "Construye herramientas declarativas para reutilizarlas en agentes y grafos."
    },
    {
        "api": "lab.Tool",
        "description": "Materializa la tool como primitiva reusable y auditable."
    },
    {
        "api": "Pydantic input/output",
        "description": "Tipa la entrada y salida para que el contrato sea verificable."
    },
    {
        "api": "ContractPolicySpec",
        "description": "Une contrato y policy para validar intencion antes de ejecutar."
    },
    {
        "api": "Tool.run",
        "description": "Ejecuta la tool de forma directa para ver su boundary aislado."
    },
    {
        "api": "human_result",
        "description": "Renderiza la salida humana sin perder la evidencia interna."
    },
    {
        "api": "manual tool chaining",
        "description": "Compone tools paso a paso para ensenar el flujo sin agente."
    }
]

lab.show({'notebook': '01_tool_api.ipynb', 'api_coverage': api_coverage})


## S?mbolos API explicados

Este notebook se alinea con `docs/API.md` y ense?a estos s?mbolos p?blicos:

- `Tool / lab.tool`: Primitiva ejecutable y decorator p?blico.
- `PublicToolRegistry`: Registro p?blico de tools cuando se necesita composici?n.
- `lab.expect`: Declaraci?n de expectativas de tools.
- `ToolExpectationValue / normalize_tool_expectation`: Normalizaci?n p?blica de expectativas.
- `validate_tool_expectation`: Validador p?blico de ejecuci?n esperada.
- `ValidationIssue / ValidationResult`: Tipos p?blicos de validaci?n.
- `AgentContract / ContractPolicySpec / RunPolicy`: Contrato y policy aplicables a tools.

